# Notebook 02 — Modern Hopfield Network

APBIO project, 2025/26.

Before plugging the Hopfield network into the DQN replay buffer, I want to
make sure it works on its own. This notebook:

1. Implements a Modern Hopfield Network as a small PyTorch module.
2. Tests associative retrieval on synthetic patterns (store some vectors,
   query with a noisy version, check whether the clean one comes back).
3. Looks at how the retrieval temperature `beta` changes behaviour.



## 1. Drive + imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/apbio_project')
(PROJECT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print('torch:', torch.__version__)

## 2. The Modern Hopfield Network

The update rule for a continuous Modern Hopfield Network is a single line:

$$
\text{retrieved} = X^\top \cdot \mathrm{softmax}(\beta \cdot X \cdot q)
$$

where `X` is the stored-pattern matrix (one pattern per row), `q` is the
query vector, and `beta` is an inverse-temperature scalar. Structurally this
is the same thing as attention: `X` are the keys/values and `q` is the
query.

I keep the stored patterns in a `torch.Tensor` buffer. For the replay buffer
use case I will overwrite old patterns when the buffer is full (FIFO), but
that logic goes in notebook 03.

In [ ]:
class HopfieldMemory(nn.Module):
    """Modern Hopfield Network for associative retrieval.

    Stores N patterns of dimension d. Given a query vector, returns a
    weighted combination of the stored patterns, sharply peaked on the
    most similar one when `beta` is large.
    """

    def __init__(self, capacity, dim, beta=1.0):
        super().__init__()
        self.capacity = capacity
        self.dim = dim
        self.beta = beta

        # Storage for the patterns and a counter for how many are valid.
        self.register_buffer('patterns', torch.zeros(capacity, dim))
        self.register_buffer('size', torch.tensor(0, dtype=torch.long))

    def store(self, x):
        """Append one or more patterns. Wraps around when full (FIFO)."""
        if x.dim() == 1:
            x = x.unsqueeze(0)
        n = x.shape[0]
        s = int(self.size.item())
        # Simple ring buffer: position = (current_size + i) mod capacity
        idx = (torch.arange(n) + s) % self.capacity
        self.patterns[idx] = x.to(self.patterns.dtype)
        self.size.fill_(min(s + n, self.capacity))

    def retrieve(self, query, return_weights=False):
        """Return a weighted combination of stored patterns for the query.

        query can be a single vector (d,) or a batch (B, d).
        """
        if query.dim() == 1:
            query = query.unsqueeze(0)

        s = int(self.size.item())
        if s == 0:
            raise RuntimeError('Hopfield memory is empty, cannot retrieve.')

        X = self.patterns[:s]               # (s, d)
        scores = self.beta * query @ X.T    # (B, s)
        weights = torch.softmax(scores, dim=-1)  # (B, s)
        retrieved = weights @ X             # (B, d)

        if return_weights:
            return retrieved, weights
        return retrieved

    def energy(self, query):
        """Hopfield energy for diagnostic plotting (lower = better match)."""
        if query.dim() == 1:
            query = query.unsqueeze(0)
        s = int(self.size.item())
        X = self.patterns[:s]
        # Ramsauer et al. eq. (4), simplified:
        # E(q) = -logsumexp(beta * X q) / beta + 0.5 * q.q + const
        lse = torch.logsumexp(self.beta * query @ X.T, dim=-1) / self.beta
        return -lse + 0.5 * (query * query).sum(dim=-1)


# Sanity: instantiate and check shapes
mem = HopfieldMemory(capacity=10, dim=4, beta=2.0)
print('empty size:', int(mem.size.item()))
mem.store(torch.randn(3, 4))
print('after storing 3:', int(mem.size.item()))
out = mem.retrieve(torch.randn(4))
print('retrieve output shape:', tuple(out.shape))

## 3. Test 1 — store 5 random patterns, query with a noisy one

The classic associative-memory sanity check. I generate 5 random 32-dim
patterns, store them, then ask the network to retrieve one by querying with
a noise-corrupted version of pattern #2. The retrieved vector should be
close to the clean pattern #2 and far from the others.

In [ ]:
N, D = 5, 32
patterns = torch.randn(N, D)

mem = HopfieldMemory(capacity=N, dim=D, beta=5.0)
mem.store(patterns)

# Query = pattern #2 + noise
target_idx = 2
noise = 0.5 * torch.randn(D)
query = patterns[target_idx] + noise

retrieved, weights = mem.retrieve(query, return_weights=True)
retrieved = retrieved.squeeze(0)
weights = weights.squeeze(0)

print('attention weights over 5 stored patterns:')
for i, w in enumerate(weights.tolist()):
    marker = '  <- target' if i == target_idx else ''
    print(f'  pattern {i}: {w:.3f}{marker}')

# Cosine similarity retrieved-vs-original
cos = torch.nn.functional.cosine_similarity(retrieved, patterns[target_idx], dim=0)
print(f'\ncosine similarity (retrieved, target) = {cos.item():.3f}')
print('(1.0 = perfect recall, 0.0 = unrelated)')

With `beta = 5` the softmax should put almost all the weight on the
correct pattern, and the retrieved vector should be almost identical to
pattern #2. Lower beta = more blurring.

## 4. Test 2 — effect of `beta`

Sweep `beta` over a range and see how cleanly the network recalls the
target pattern. Small beta = soft mixing, large beta = sharp winner-takes-all.

In [ ]:
betas = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
cos_scores = []

for b in betas:
    m = HopfieldMemory(capacity=N, dim=D, beta=b)
    m.store(patterns)
    r = m.retrieve(query).squeeze(0)
    c = torch.nn.functional.cosine_similarity(r, patterns[target_idx], dim=0).item()
    cos_scores.append(c)
    print(f'beta = {b:>5.1f}  cosine sim = {c:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(betas, cos_scores, marker='o')
ax.set_xscale('log')
ax.set_xlabel('beta (log scale)')
ax.set_ylabel('cosine similarity retrieved vs. target')
ax.set_title('Hopfield recall sharpness vs. inverse temperature')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'figures' / 'hopfield_beta_sweep.png', dpi=150)
plt.show()

## 5. Test 3 — capacity vs. retrieval quality

Storing more patterns makes them harder to tell apart. I store an
increasing number of random patterns (always with the same beta) and
measure average recall accuracy over 20 random queries per run.

In [ ]:
D = 32
NOISE = 0.5
BETA = 5.0
capacities = [5, 10, 20, 50, 100, 200, 500]

mean_cos = []
for cap in capacities:
    P = torch.randn(cap, D)
    m = HopfieldMemory(capacity=cap, dim=D, beta=BETA)
    m.store(P)

    scores = []
    for _ in range(20):
        i = np.random.randint(cap)
        q = P[i] + NOISE * torch.randn(D)
        r = m.retrieve(q).squeeze(0)
        scores.append(torch.nn.functional.cosine_similarity(r, P[i], dim=0).item())
    mean_cos.append(np.mean(scores))

for cap, c in zip(capacities, mean_cos):
    print(f'capacity = {cap:>4d}   mean cos = {c:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(capacities, mean_cos, marker='o')
ax.set_xscale('log')
ax.set_xlabel('stored patterns (log scale)')
ax.set_ylabel('mean cosine similarity over 20 queries')
ax.set_title(f'Recall quality vs. memory load (d={D}, noise={NOISE}, beta={BETA})')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'figures' / 'hopfield_capacity.png', dpi=150)
plt.show()

## 6. Quick takeaways

- The retrieval rule is a one-liner (`softmax(beta * X @ q) @ X`) but it
  behaves like a full associative memory.
- Higher `beta` = sharper recall. Typical values for our use case will be
  between 1 and 10.
- Recall stays high even at modest load (100+ patterns in 32 dimensions),
  which is comfortably more than what CartPole will need.

Next: in notebook 03 I wrap this into a `HopfieldReplayBuffer` that
plugs into SB3's DQN.